# Baseline-defense comparison: JANUS-FL's rejection vs. the standard robust-aggregation toolkit

Closes brutal-review gap #1: "no baseline comparison against
established robust-aggregation defenses... the comparison is 'our
system with the attack vs. our system without it' -- never 'our
system vs. a competing defense under the same attack.'"

A grid of **(aggregation rule) x (attack)**, same model (`tiny_cnn`),
same data (MNIST, case1 IID split), same 50 clients, same 20%
Byzantine fraction (10 cheaters), same 15 rounds, same
per-client local-training step everywhere (every cell calls the real,
unmodified `fl/client.py::local_train_delta`) -- final-round test
accuracy recorded for every cell.

**Aggregation rules**: `mean` (no defense -- the honest baseline),
`coordinate_median`, `trimmed_mean`, `krum`, `multi_krum`
(`baselines/aggregation_rules.py`; Blanchard et al. 2017, Yin et al. 2018).

**Attacks**: sign-flip and Gaussian-noise -- the two classic
unbounded-magnitude attacks the Krum/trimmed-mean/median literature
itself benchmarks against. Deliberately NOT scale-to-threshold or
low-and-slow here, since those specifically target JANUS-FL's OWN
R_norm threshold, which a plaintext comparison loop has no equivalent
of -- those two already got their honest, real treatment against the
actual threshold in `RUN_ATTACKS.ipynb`'s A1 gate.

**What this does NOT claim**: every rule here runs over PLAINTEXT
deltas -- none of this is JANUS-FL's own secure-aggregation layer.
Coordinate-median and Krum both need plaintext visibility to compute
their order statistics/pairwise distances, which secure aggregation
specifically hides from the server; that tension is real, not papered
over. This notebook answers "how does JANUS-FL's rejection-based
defense compare to the field's standard toolkit under the same
attacks and scale" -- not "could these rules be dropped into the real
pipeline unmodified."


In [ ]:
# 1. Clone/pull the repo. (No Drive mount -- checkpoints go through
# git instead, see the git-checkpoint-setup cell further down and this
# notebook's own intro markdown for why.)
import os

if not os.path.exists('/content/FL-ZKP'):
    !git clone https://github.com/YashasweeNarayan/FL-ZKP.git /content/FL-ZKP
else:
    !cd /content/FL-ZKP && git fetch -q origin main && git reset -q --hard origin/main
%cd /content/FL-ZKP


In [ ]:
# 2. Install the EC-dependent toolchain (rbcl for ristretto255, torch/torchvision, flwr-datasets).
!pip install -q rbcl
!pip install -q pynacl
!pip install -q flwr-datasets[vision]
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121


**If this is the first time cell 2 has installed anything in this
session, restart the kernel now: Runtime -> Restart session.** Then
continue from cell 3 below (NOT from cell 1 -- Drive stays mounted and
the repo stays cloned across a restart, only the Python process
resets, so re-running cell 1 would be harmless but pointless).


In [ ]:
# 3. Sanity-check the install.
import sys
sys.path.insert(0, '/content/FL-ZKP')
import rbcl
assert hasattr(rbcl, "crypto_core_ristretto255_from_hash"), (
    "rbcl imported but is missing crypto_core_ristretto255_from_hash -- "
    "see crypto/ristretto.py's import fallback and FIX.md F8."
)
import torch
print("rbcl OK. torch OK, CUDA available:", torch.cuda.is_available())


In [ ]:
# 4. Load config.
import yaml
with open('configs/mvp_3day.yaml') as f:
    CFG = yaml.safe_load(f)
print(CFG)


## Data + model-building helper

In [ ]:
# 5. Load MNIST, stratified-IID split across clients.
# NOTE: configs/mvp_3day.yaml's datasets.mnist.source is "csv" (reads a
# Drive-cached CSV that this run's Drive doesn't have) -- that path is
# for whoever already staged the CSV. This run instead forces
# source="torchvision", which downloads MNIST directly over the network,
# no Drive dependency at all. Deliberately NOT changed in the shared
# config file, since other runs may rely on the csv path being default.
from fl.data_iid import prepare_case1

case1_data = prepare_case1(seed=CFG['seed'], source="torchvision")
print("clients:", len(case1_data['client_train']))
print("server_root size:", len(case1_data['server_root'].labels))
print("global_val size:", len(case1_data['global_val'].labels))


In [ ]:
n_clients = CFG['protocol']['n_clients']
from fl.models import build_model, flatten_state_dict
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('n_clients:', n_clients, '| device:', device)


## Grid setup -- aggregation rules x attacks x Byzantine fraction

In [ ]:
import sys
sys.path.insert(0, '/content/FL-ZKP')
import numpy as np
import torch
from baselines.plain_fedavg_loop import run_plain_fedavg_loop
from baselines.aggregation_rules import AGGREGATION_RULES
from attacks.baseline_norm_attacks import craft_sign_flip, craft_gaussian_noise

N_BYZANTINE = int(round(0.2 * n_clients))
CHEATER_INDICES = set(range(N_BYZANTINE))  # first N_BYZANTINE client indices, same convention FIX.md's attack notebooks use

ATTACKS = {
    "no_attack": None,
    "sign_flip": lambda delta: craft_sign_flip(delta),
    "gaussian_noise": lambda delta: craft_gaussian_noise(delta, sigma=1.0, rng=np.random.default_rng(CFG['seed'])),
}

print(f"grid: {len(AGGREGATION_RULES)} aggregation rules x {len(ATTACKS)} attacks, "
      f"{N_BYZANTINE}/{n_clients} Byzantine clients, 15 rounds each")


In [ ]:
# Shared eval_fn for every grid cell -- forward-only pass over the
# official MNIST test split, same split headline accuracy is always
# reported against elsewhere in this project. Each grid cell gets its
# OWN freshly-initialized model (built inside the loop below), so this
# closure reads whichever model run_plain_fedavg_loop is currently
# training against.
_eval_images, _eval_labels = case1_data['test']

def make_eval_fn():
    def _eval_fn(model):
        model.eval()
        correct = 0
        with torch.no_grad():
            for i in range(0, len(_eval_images), 512):
                bi = _eval_images[i:i + 512].to(device)
                bl = _eval_labels[i:i + 512].to(device)
                correct += (model(bi).argmax(dim=1) == bl).sum().item()
        acc = correct / len(_eval_images)
        return acc, None
    return _eval_fn


## Run the grid -- 15 rounds per cell

In [ ]:
# The grid itself -- 5 aggregation rules x 3 attacks = 15 cells, each
# 15 rounds, each a FRESH model (no weight-sharing across cells).
results = {}
for rule_name, rule_fn in AGGREGATION_RULES.items():
    for attack_name, craft_fn in ATTACKS.items():
        grid_model = build_model('tiny_cnn', num_classes=10, in_channels=1).to(device)
        grid_flat, _ = flatten_state_dict(grid_model)
        grid_dim = grid_flat.numel()

        cheater_idx = CHEATER_INDICES if attack_name != "no_attack" else set()
        result = run_plain_fedavg_loop(
            model=grid_model,
            model_dim=grid_dim,
            client_datasets=case1_data['client_train'][:n_clients],
            rounds=15,
            lr=CFG['local_training']['lr'],
            epochs=CFG['local_training']['local_epochs'],
            batch_size=CFG['local_training']['batch_size'],
            aggregator=rule_fn,
            n_byzantine=N_BYZANTINE,
            cheater_indices=cheater_idx,
            craft_fn=craft_fn,
            eval_fn=make_eval_fn(),
            seed=CFG['seed'],
        )
        final_acc = result.history[-1].test_acc
        results[(rule_name, attack_name)] = final_acc
        print(f"  {rule_name:>18} x {attack_name:<15}: final test acc = {final_acc:.4f}")


## Results table

In [ ]:
# Save the results table.
import csv

rule_names = list(AGGREGATION_RULES.keys())
attack_names = list(ATTACKS.keys())

print()
print("| aggregation rule".ljust(20) + "".join(f"| {a:<15}" for a in attack_names) + "|")
print("|" + "-"*19 + ("|" + "-"*16) * len(attack_names) + "|")
for rule_name in rule_names:
    row = f"| {rule_name:<18}"
    for attack_name in attack_names:
        row += f"| {results[(rule_name, attack_name)]:<15.4f}"
    print(row + "|")

with open('/content/FL-ZKP/baseline_defenses_results.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["aggregation_rule"] + attack_names)
    for rule_name in rule_names:
        writer.writerow([rule_name] + [results[(rule_name, a)] for a in attack_names])
print("\nsaved: /content/FL-ZKP/baseline_defenses_results.csv")
